# Tree-Guided Retrieval Check

This Layer 4A checkpoint selects document regions from the cached PageIndex Flash tree and obtains authoritative text from PyMuPDF extraction. It does not generate final answers.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').is_dir():
    raise RuntimeError('Run this notebook from the ConversIQ project or notebooks directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CACHE_DIR, PAGEINDEX_MODEL, SAMPLE_DOCS_DIR
from src.extraction import generate_doc_id
from src.graph import get_graph
from src.retriever import generate_answer, retrieve_document

pdf_path = SAMPLE_DOCS_DIR / 'harry_potter.pdf'
if not pdf_path.is_file():
    raise FileNotFoundError(f'Expected the manual test PDF at: {pdf_path}')

doc_id = generate_doc_id(pdf_path)
cache_path = CACHE_DIR / f'{doc_id}.json'
if not cache_path.is_file():
    raise FileNotFoundError(f'Build the PageIndex tree first: {cache_path}')

print(f'PDF path: {pdf_path}')
print(f'Document ID: {doc_id}')
print(f'Retrieval model: {PAGEINDEX_MODEL}')
print(f'Cache path: {cache_path}')

PDF path: D:\conversIQ_v3\data\sample_docs\harry_potter.pdf
Document ID: 880c366958602a7e
Retrieval model: groq/openai/gpt-oss-20b
Cache path: D:\conversIQ_v3\cache\trees\880c366958602a7e.json


## Manual retrieval queries

Each query makes one retrieval-selection LLM call. `source_text` is constructed from the selected extracted pages; no final answer is generated.

In [2]:
def preview(text: str, limit: int = 1_200) -> str:
    normalized = ' '.join(text.split())
    if not normalized:
        return '<empty source text>'
    return normalized[:limit] + ('…' if len(normalized) > limit else '')

def show_retrieval(query: str) -> None:
    result = retrieve_document(query, doc_id, pdf_path)
    print('QUESTION')
    print(query)
    print('\nSELECTED NODES')
    if result.selected_nodes:
        for node in result.selected_nodes:
            print(f'- {node.node_id} | {node.title} | pages {node.start_index}-{node.end_index}')
    else:
        print('<none>')
    print('\nRELEVANT PAGES')
    print(list(result.relevant_pages) if result.relevant_pages else '<none>')
    print('\nSOURCE PAGES')
    if result.source_pages:
        for page in result.source_pages:
            print(f'- Page {page.page_number} | {len(page.text)} extracted characters')
    else:
        print('<none>')
    print('\nSOURCE TEXT PREVIEW')
    print(preview(result.source_text))
    print(f'\nSelected nodes: {len(result.selected_nodes)}')
    print(f'Relevant pages: {len(result.relevant_pages)}')
    print(f'Source pages: {len(result.source_pages)}')
    print(f'Source text characters: {len(result.source_text)}')
    print('\n' + '=' * 80 + '\n')
    return result

manual_queries = (
    'What are the functional requirements?',
    'what happens in the riddle house',
    'What is the capital of France?',
)

retrieval_results = {
    manual_query: show_retrieval(manual_query)
    for manual_query in manual_queries
}

QUESTION
What are the functional requirements?

SELECTED NODES
<none>

RELEVANT PAGES
<none>

SOURCE PAGES
<none>

SOURCE TEXT PREVIEW
<empty source text>

Selected nodes: 0
Relevant pages: 0
Source pages: 0
Source text characters: 0


QUESTION
what happens in the riddle house

SELECTED NODES
- 0001 | The Riddle House | pages 7-22

RELEVANT PAGES
[7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]

SOURCE PAGES
- Page 7 | 1111 extracted characters
- Page 8 | 1633 extracted characters
- Page 9 | 1650 extracted characters
- Page 10 | 1634 extracted characters
- Page 11 | 1723 extracted characters
- Page 12 | 1683 extracted characters
- Page 13 | 1644 extracted characters
- Page 14 | 1626 extracted characters
- Page 15 | 1593 extracted characters
- Page 16 | 1688 extracted characters
- Page 17 | 1599 extracted characters
- Page 18 | 1685 extracted characters
- Page 19 | 1537 extracted characters
- Page 20 | 1519 extracted characters
- Page 21 | 764 extracted characters
- Page 22

## Layer 4B — Grounded Answer Generation

This section reuses the Layer 4A retrieval results above, avoiding duplicate retrieval-selector calls. `generate_answer()` calls the answer model only when retrieved source text and page references exist. The answer prompt now requests a short conversational paragraph with page-only citations, not Markdown lists or invented line numbers.

In [3]:
def show_grounded_answer(query: str) -> None:
    result = retrieval_results[query]
    print('QUESTION')
    print(query)
    print('\nSELECTED NODES')
    print([node.node_id for node in result.selected_nodes] or '<none>')
    print('RELEVANT PAGES')
    print(list(result.relevant_pages) or '<none>')
    print('SOURCE TEXT PREVIEW')
    print(preview(result.source_text, limit=900))
    answer = generate_answer(query, result.source_text, result.relevant_pages)
    print('\nGROUNDED ANSWER')
    print(answer)
    print('\n' + '=' * 80 + '\n')

if 'retrieval_results' not in globals():
    retrieval_results = {
        manual_query: retrieve_document(manual_query, doc_id, pdf_path)
        for manual_query in manual_queries
    }

for manual_query in manual_queries:
    show_grounded_answer(manual_query)

QUESTION
What are the functional requirements?

SELECTED NODES
<none>
RELEVANT PAGES
<none>
SOURCE TEXT PREVIEW
<empty source text>

GROUNDED ANSWER
The document does not contain relevant information to answer this question.


QUESTION
what happens in the riddle house

SELECTED NODES
['0001']
RELEVANT PAGES
[7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]
SOURCE TEXT PREVIEW
[Page 7] Chapter 1 The Riddle House The villagers of Little Hangleton still called it “the Riddle House,” even though it had been many years since the Riddle family had lived there. It stood on a hill overlooking the village, some of its windows boarded, tiles missing from its roof, and ivy spreading unchecked over its face. Once a ﬁne-looking manor, and easily the largest and grandest building for miles around, the Riddle House was now damp, derelict, and unoccupied. The Little Hangletons all agreed that the old house was “creepy.” Half a century ago, something strange and horrible had happened there,

## Layer 5 — LangGraph Orchestration

The graph invokes `START → load_tree → retrieve → generate_answer → END`. It uses the existing cached tree, Layer 4 retrieval, and Layer 4B answer function without duplicating their logic.

In [4]:
workflow = get_graph()
print('Workflow: START → load_tree → retrieve → generate_answer → END')

def show_graph_run(query: str) -> None:
    graph_input = {
        'doc_id': doc_id,
        'file_path': pdf_path,
        'query': query,
    }
    final_state = workflow.invoke(graph_input)

    print('\nGRAPH INPUT')
    print({key: str(value) for key, value in graph_input.items()})
    print('\nGRAPH OUTPUT')
    print({
        'tree_loaded': isinstance(final_state.get('tree'), dict),
        'selected_node_ids': [node.node_id for node in final_state.get('selected_nodes', ())],
        'source_page_count': len(final_state.get('source_pages', ())),
        'source_text_characters': len(final_state.get('source_text', '')),
        'answer_present': bool(final_state.get('answer')),
    })
    print('\nRELEVANT PAGES')
    print(list(final_state.get('relevant_pages', ())) or '<none>')
    print('\nSOURCE TEXT PREVIEW')
    print(preview(final_state.get('source_text', ''), limit=900))
    print('\nANSWER')
    print(final_state.get('answer', '<missing>'))
    print('\n' + '=' * 80)

show_graph_run('explain the architecture of the system')
show_graph_run('What are the non function requirements?')

Workflow: START → load_tree → retrieve → generate_answer → END

GRAPH INPUT
{'doc_id': '880c366958602a7e', 'file_path': 'D:\\conversIQ_v3\\data\\sample_docs\\harry_potter.pdf', 'query': 'explain the architecture of the system'}

GRAPH OUTPUT
{'tree_loaded': True, 'selected_node_ids': [], 'source_page_count': 0, 'source_text_characters': 0, 'answer_present': True}

RELEVANT PAGES
<none>

SOURCE TEXT PREVIEW
<empty source text>

ANSWER
The document does not contain relevant information to answer this question.


GRAPH INPUT
{'doc_id': '880c366958602a7e', 'file_path': 'D:\\conversIQ_v3\\data\\sample_docs\\harry_potter.pdf', 'query': 'What are the non function requirements?'}

GRAPH OUTPUT
{'tree_loaded': True, 'selected_node_ids': [], 'source_page_count': 0, 'source_text_characters': 0, 'answer_present': True}

RELEVANT PAGES
<none>

SOURCE TEXT PREVIEW
<empty source text>

ANSWER
The document does not contain relevant information to answer this question.



## Latency Diagnostics

This checkpoint measures each operation once. The direct total is calculated from the separately measured retrieval and answer calls to avoid duplicating those LLM requests. A single graph invocation provides the end-to-end orchestration comparison.

In [5]:
from time import perf_counter

from src.extraction import extract_pdf
from src.indexer import load_tree
from src.retriever import _flatten_tree

diagnostic_query = 'What happens at the Riddle House?'

tree_started = perf_counter()
diagnostic_tree = load_tree(doc_id)
load_tree_latency = perf_counter() - tree_started
candidates = _flatten_tree(diagnostic_tree)
candidate_list = '\n'.join(
    f'{node.node_id} | {node.title} | pages {node.start_index}-{node.end_index}'
    for node in candidates
)

extraction_started = perf_counter()
extracted_document = extract_pdf(pdf_path)
pdf_extraction_latency = perf_counter() - extraction_started

retrieval_started = perf_counter()
diagnostic_result = retrieve_document(diagnostic_query, doc_id, pdf_path)
retrieval_latency = perf_counter() - retrieval_started

answer_started = perf_counter()
diagnostic_answer = generate_answer(
    diagnostic_query,
    diagnostic_result.source_text,
    diagnostic_result.relevant_pages,
)
answer_latency = perf_counter() - answer_started
direct_total_latency = retrieval_latency + answer_latency

graph_compile_started = perf_counter()
workflow = get_graph()
graph_compile_latency = perf_counter() - graph_compile_started
graph_started = perf_counter()
graph_state = workflow.invoke({
    'doc_id': doc_id,
    'file_path': pdf_path,
    'query': diagnostic_query,
})
graph_total_latency = perf_counter() - graph_started
graph_node_timings = graph_state.get('node_timings', {})
graph_node_call_counts = graph_state.get('node_call_counts', {})

print('LATENCY RESULTS')
print(f'Cached load_tree latency: {load_tree_latency:.2f} s')
print(f'PDF extraction latency: {pdf_extraction_latency:.2f} s')
print(f'Retrieval latency: {retrieval_latency:.2f} s')
print(f'Answer generation latency: {answer_latency:.2f} s')
print(f'Direct total latency (retrieval + answer): {direct_total_latency:.2f} s')
print(f'Graph total latency: {graph_total_latency:.2f} s')
print(f'Graph compilation latency: {graph_compile_latency:.2f} s')
print(f'Approx. selector time (retrieval - standalone extraction): {max(0, retrieval_latency - pdf_extraction_latency):.2f} s')

print('\nGRAPH NODE TIMINGS')
for node_name in ('load_tree', 'retrieve', 'generate_answer'):
    print(
        f'{node_name}: {graph_node_timings.get(node_name, 0.0):.2f} s '
        f'(calls: {graph_node_call_counts.get(node_name, 0)})'
    )
print(f'GRAPH TOTAL: {graph_total_latency:.2f} s')
print(f'DIRECT TOTAL: {direct_total_latency:.2f} s')

print('\nREQUEST SIZE / CONFIGURATION')
print(f'Retrieval selector model: {PAGEINDEX_MODEL}')
print(f'Retrieval candidate nodes: {len(candidates)}')
print(f'Retrieval candidate-list characters: {len(candidate_list)}')
print('Retrieval structured output: json_mode')
print('Retrieval temperature: 0; streaming: False; retries: ChatGroq default (2)')
print('Answer model: openai/gpt-oss-120b')
print('Answer temperature: 0; max_tokens: 500; reasoning: unset; streaming: False; retries: ChatGroq default (2)')
print(f'Answer source-text characters: {len(diagnostic_result.source_text)}')
print(f'Answer relevant pages: {len(diagnostic_result.relevant_pages)}')
print(f'Graph selected nodes: {[node.node_id for node in graph_state.get("selected_nodes", ())]}')
print(f'Graph relevant pages: {len(graph_state.get("relevant_pages", ()))}')
print(f'Graph answer characters: {len(graph_state.get("answer", ""))}')

LATENCY RESULTS
Cached load_tree latency: 0.00 s
PDF extraction latency: 0.09 s
Retrieval latency: 2.27 s
Answer generation latency: 42.18 s
Direct total latency (retrieval + answer): 44.45 s
Graph total latency: 50.57 s
Graph compilation latency: 0.00 s
Approx. selector time (retrieval - standalone extraction): 2.18 s

GRAPH NODE TIMINGS
load_tree: 0.00 s (calls: 1)
retrieve: 1.72 s (calls: 1)
generate_answer: 48.85 s (calls: 1)
GRAPH TOTAL: 50.57 s
DIRECT TOTAL: 44.45 s

REQUEST SIZE / CONFIGURATION
Retrieval selector model: groq/openai/gpt-oss-20b
Retrieval candidate nodes: 26
Retrieval candidate-list characters: 934
Retrieval structured output: json_mode
Retrieval temperature: 0; streaming: False; retries: ChatGroq default (2)
Answer model: openai/gpt-oss-120b
Answer temperature: 0; max_tokens: 500; reasoning: unset; streaming: False; retries: ChatGroq default (2)
Answer source-text characters: 24338
Answer relevant pages: 16
Graph selected nodes: ['0001']
Graph relevant pages: 16
